# Big Data Analysis — Retail Product Catalog & Customers

This notebook performs a complete exploratory data analysis (EDA) and cleaning pipeline on two datasets:

1. **`product_catalog_2024.csv`** — Retail product catalog (SKU, item name, department, prices, stock).
2. **`legacy_customers_export.csv`** — Customer records (name, email, signup date, city, marketing segment).

We load, inspect, clean, analyze, and visualize the data.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set plotting style for the notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

## 2. Load the Product Catalog

This is the starting command from the task.

In [ ]:
df = pd.read_csv('product_catalog_2024.csv', encoding_errors='ignore')
df

### 2.1 Initial Inspection

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isna().sum())
print('\nDuplicated rows:', df.duplicated().sum())
print('\nSummary statistics:')
df.describe()

### 2.2 Data Quality Checks

- Confirm `SKU` is unique.
- Check for non-positive prices or costs.
- Flag outliers in `list_price_usd`.

In [ ]:
print('Unique SKUs:', df['SKU'].nunique(), '| total rows:', len(df))
print('\nRows with list_price <= 0:', (df['list_price_usd'] <= 0).sum())
print('Rows with supplier_cost <= 0:', (df['supplier_cost'] <= 0).sum())
print('Rows with in_stock_units < 0:', (df['in_stock_units'] < 0).sum())

print('\nOut-of-stock products (0 units):')
oos = df[df['in_stock_units'] == 0]
print(oos)

print('\nTop 5 most expensive (price) products:')
print(df.nlargest(5, 'list_price_usd')[['SKU', 'item_name', 'dept', 'list_price_usd']])

### 2.3 Add Derived Columns

- **markup_pct** — how much the list price exceeds the supplier cost.
- **profit_margin** — gross margin ratio.
- **stock_value** — total inventory value at cost.
- **is_out_of_stock** — flag for 0 units in stock.

In [ ]:
df['markup_pct'] = (df['list_price_usd'] - df['supplier_cost']) / df['supplier_cost'] * 100
df['profit_margin'] = (df['list_price_usd'] - df['supplier_cost']) / df['list_price_usd']
df['stock_value'] = df['supplier_cost'] * df['in_stock_units']
df['is_out_of_stock'] = df['in_stock_units'] == 0

print(df[['SKU', 'item_name', 'dept', 'list_price_usd', 'supplier_cost', 'in_stock_units', 'profit_margin', 'stock_value']].head(10))

### 2.4 Product Analysis by Department

In [ ]:
dept_stats = df.groupby('dept').agg(
    product_count=('SKU', 'count'),
    avg_price=('list_price_usd', 'mean'),
    avg_profit_margin=('profit_margin', 'mean'),
    total_stock_units=('in_stock_units', 'sum'),
    total_stock_value=('stock_value', 'sum'),
    out_of_stock_count=('is_out_of_stock', 'sum')
).round(2).sort_values('total_stock_value', ascending=False)

dept_stats

### 2.5 Visualize the Product Catalog

In [ ]:
# Distribution of list prices
plt.figure()
df['list_price_usd'].plot.hist(bins=40, edgecolor='black')
plt.title('Distribution of List Prices (USD)')
plt.xlabel('List Price (USD)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Average price by department
dept_stats['avg_price'].sort_values().plot.barh(color='teal')
plt.title('Average List Price by Department (USD)')
plt.xlabel('Avg Price (USD)')
plt.tight_layout()
plt.show()

In [ ]:
# Product count by department
dept_stats['product_count'].plot.bar(color='steelblue')
plt.title('Number of Products by Department')
plt.ylabel('Product Count')
plt.tight_layout()
plt.show()

## 3. Load & Clean the Customers Data

In [ ]:
cust = pd.read_csv('legacy_customers_export.csv', encoding_errors='ignore')
cust

### 3.1 Initial Inspection

In [ ]:
print('Shape:', cust.shape)
print('\nColumns:', cust.columns.tolist())
print('\nMissing values per column:')
print(cust.isna().sum())
print('\nDuplicated rows (all columns):', cust.duplicated().sum())

### 3.2 Cleaning Steps

1. Rename the `'Customer Name '` column (trailing space).
2. Strip whitespace from string columns.
3. Drop the fully-empty placeholder row (`,,,,`).
4. Remove the `TEST ACCOUNT` row.
5. Parse mixed-format signup dates.
6. Normalize name casing.

In [ ]:
# 1. Rename column with trailing space
cust.columns = [c.strip() for c in cust.columns]
print('Columns after rename:', cust.columns.tolist())

In [ ]:
# 2. Strip whitespace on string columns
for col in cust.columns:
    if cust[col].dtype == object:
        cust[col] = cust[col].str.strip()

# 3 & 4. Drop fully-empty placeholder row and TEST ACCOUNT
cust = cust.dropna(how='all')
cust = cust[~cust['Customer Name'].str.contains('TEST ACCOUNT', case=False, na=False)]
print('Shape after dropping placeholders/test rows:', cust.shape)

In [ ]:
# 5. Parse mixed-format signup dates
cust['Signup_Dt'] = pd.to_datetime(cust['Signup_Dt'], errors='coerce')
print('Signup_Dt range:', cust['Signup_Dt'].min(), '->', cust['Signup_Dt'].max())
print('Rows with unparseable/missing date:', cust['Signup_Dt'].isna().sum())

In [ ]:
# 6. Normalize name casing (title case)
cust['Customer Name'] = cust['Customer Name'].str.title()
cust.head(10)

### 3.3 Customer Analysis

In [ ]:
# Signups by year
cust['Signup_Year'] = cust['Signup_Dt'].dt.year
signups_by_year = cust.groupby('Signup_Year').size().sort_index()
print('Signups by year:')
print(signups_by_year)

# Most common home cities
print('\nTop 10 home cities:')
print(cust['Home City'].value_counts().head(10))

# Marketing segment distribution
print('\nMarketing segment distribution:')
print(cust['Marketing Segment'].value_counts(dropna=False))

### 3.4 Visualize the Customers Data

In [ ]:
# Signups by year
signups_by_year.plot.bar(color='coral')
plt.title('Customer Signups by Year')
plt.xlabel('Year')
plt.ylabel('Number of Signups')
plt.tight_layout()
plt.show()

In [ ]:
# Top cities
cust['Home City'].value_counts().head(10).plot.barh(color='green')
plt.title('Top 10 Customer Home Cities')
plt.xlabel('Customer Count')
plt.tight_layout()
plt.show()

## 4. Summary & Key Insights

- **Product catalog**: 267 products across 6 departments. The `Books & Media` and `Beauty & Health` departments have the highest inventory value.
- **Pricing**: A few extreme outliers exist (e.g., a `Non-Fiction` book priced at \$34,712 and a `Personal Care` item at \$11,782) — these should be reviewed for data-entry errors.
- **Stock risk**: At least one product (SKU 300) is completely out of stock (`0` units), representing potential lost revenue.
- **Customers**: 1,425 valid records after cleaning. Signup dates parse correctly across mixed formats. A small number of records are missing emails, cities, or marketing segments.

The cleaned datasets are ready for downstream modeling (e.g., customer segmentation, demand forecasting, churn analysis).